# Importaciones

In [ ]:
import os
import sys
import gc
import json
import time
import math
import numpy as np
import pandas as pd
import faiss
import igraph as ig
import networkx as nx
import plotly.graph_objects as go
from collections import Counter, defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import CountVectorizer

# Configuración de reproducibilidad con semilla estricta
SEMILLA = 42
np.random.seed(SEMILLA)
import random
random.seed(SEMILLA)
ig.set_random_number_generator(random.Random(SEMILLA))

# Rutas de datos
PARQUET_FILE = "../RMHD_con_emociones_new.parquet" if os.path.exists("../RMHD_con_emociones_new.parquet") else "RMHD_con_emociones_new.parquet"
EMB_FILE = "../embeddings.npy" if os.path.exists("../embeddings.npy") else "embeddings.npy"

# Dimensiones y factores afectivos
FACTOR_COLS = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
SUBREDDITS_OBJETIVO = ["Anxiety", "SuicideWatch", "depression"]

MONTH_TO_Q = {
    'jan': 'Q1', 'feb': 'Q1', 'mar': 'Q1',
    'apr': 'Q2', 'may': 'Q2', 'jun': 'Q2',
    'jul': 'Q3', 'aug': 'Q3', 'sep': 'Q3',
    'oct': 'Q4', 'nov': 'Q4', 'dec': 'Q4'
}

print(f"[OK] Entorno cargado exitosamente. Semilla={SEMILLA}.")


[OK] Entorno cargado exitosamente. Semilla=42.


# Carga de datos


In [ ]:
# Carga de metadatos de Parquet
print("[*] Verificando integridad de datos...")
df_full = pd.read_parquet(PARQUET_FILE)
print(f"Dimensiones de Parquet: {df_full.shape[0]:,} filas x {df_full.shape[1]} columnas.")

# Mapeo de embeddings
emb_full = np.load(EMB_FILE, mmap_mode="r")
print(f"Dimensiones de Embeddings: {emb_full.shape[0]:,} filas x {emb_full.shape[1]} dimensiones.")

# Se verifica que el archivo de embeddings esté alineado con el dataset
assert len(df_full) == len(emb_full), "¡Error crítico! Desalineación entre Parquet y Embeddings."
assert not np.isnan(emb_full[:10000]).any(), "¡Error! Valores NaN detectados en la muestra de embeddings."

print("[OK] Alineación 1:1 e integridad de datos completamente verificadas.")

# Generalidades


In [18]:
# Distribución de comentarios por subreddit
print("Distribución por subreddit:")
display(df_full["subreddit"].value_counts().to_frame())

# Secuencia cronológica estricta de trimestres
def ordenar_periodos_cronologicamente(lista_periodos):
    return sorted(list(set(lista_periodos)), key=lambda x: (int(x[:4]), int(x[-1])))

periodos_cronologicos = ordenar_periodos_cronologicamente(df_full["period"].dropna().unique())
print(f"Secuencia cronológica ({len(periodos_cronologicos)} trimestres continuos):")
print(periodos_cronologicos)

# Tabla cruzada de volumen trimestral
tabla_temporal = pd.crosstab(df_full["period"], df_full["subreddit"]).loc[periodos_cronologicos]
display(tabla_temporal)


Distribución por subreddit:


,count
subreddit,
depression,620248
SuicideWatch,469735
Anxiety,278631


Secuencia cronológica (15 trimestres continuos):
['2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3']


subreddit,Anxiety,SuicideWatch,depression
period,,,
2019Q1,14908,16419,39713
2019Q2,14188,18870,43808
2019Q3,15735,21726,46628
2019Q4,14764,22838,34927
2020Q1,16784,25199,48791
2020Q2,18289,28942,44981
2020Q3,19594,31933,43223
2020Q4,20109,39040,49111
2021Q1,18218,34570,41827


# Búsqueda en Grilla
Se realizó una búsqueda sistemática en grilla evaluando:
- **Umbral de similitud semántica ($\epsilon$):** `0.92, 0.95, 0.97`
- **Vecinos más cercanos ($k$):** `20, 30, 50`
- **Algoritmo de detección de comunidades:** Comparación entre **Louvain** y **Leiden** 
- **Parámetro de resolución:** `0.05, 0.08, 0.10, 0.15, 0.20`
- **Agrupamiento macrocomunitario:** $K \in \{8, 10, 12, 14\}$

In [19]:
# Carga del reporte comparativo de hiperparámetros
archivo_reporte_csv = "reporte_optimizacion_hiperparametros_sna.xlsx"
if os.path.exists(archivo_reporte_csv):
    df_grid = pd.read_excel(archivo_reporte_csv)
    print("Resumen de las mejores configuraciones por subreddit:")
    df_ganadores = df_grid[df_grid["Fase"].str.contains("Validación Definitiva", na=False)]
    display(df_ganadores[["Subreddit", "N_Total", "k", "Threshold", "Aristas", "Modularidad", "Comunidades_Totales", "Cobertura_Sustancial (%)", "Coherencia_Intra", "Separacion_Inter"]])
else:
    print("[!] Archivo de reporte no encontrado. Utilizando parámetros ganadores preestablecidos.")

CONFIGURACIONES_GANADORAS = {
    "Anxiety":      {"k": 30, "threshold": 0.95, "resolution": 0.10, "target_k": 10},
    "SuicideWatch": {"k": 30, "threshold": 0.95, "resolution": 0.10, "target_k": 10},
    "depression":   {"k": 30, "threshold": 0.95, "resolution": 0.10, "target_k": 12}
}


Resumen de las mejores configuraciones por subreddit:


,Subreddit,N_Total,k,Threshold,Aristas,Modularidad,Comunidades_Totales,Cobertura_Sustancial (%),Coherencia_Intra,Separacion_Inter
152,Anxiety,278631,30,0.95,5829378,0.7507,10,98.42,0.8349,1.0555
153,SuicideWatch,469735,30,0.95,9711095,0.7290,10,98.97,0.8512,1.0720
154,depression,620248,30,0.95,12894839,0.8216,12,98.93,0.8516,1.0560


# Construcción de Grafos

In [20]:
def aplicar_regla_neutral_cortos(G):
    txt_attr = "comentario_completo" if "comentario_completo" in G.vertex_attributes() else ("text" if "text" in G.vertex_attributes() else None)
    if txt_attr is None:
        return G
    textos = G.vs[txt_attr]
    es_corto = [len(str(t or "").split()) < 5 for t in textos]
    if "factor" in G.vertex_attributes():
        G.vs["factor"] = ["neutral" if corto else f for corto, f in zip(es_corto, G.vs["factor"])]
    if "emocion" in G.vertex_attributes():
        G.vs["emocion"] = ["neutral" if corto else e for corto, e in zip(es_corto, G.vs["emocion"])]
    for f_col in FACTOR_COLS:
        if f_col in G.vertex_attributes():
            val_corto = 1.0 if f_col == "neutral" else 0.0
            G.vs[f_col] = [val_corto if corto else v for corto, v in zip(es_corto, G.vs[f_col])]
    return G

def cargar_o_construir_grafo(sub, cfg, df_full, emb_full):
    pkl_file = f"Grafo_{sub}_best.pkl"
    topics_file = f"topics_{sub}_best.json"
    
    if os.path.exists(pkl_file) and os.path.exists(topics_file):
        print(f"[+] r/{sub}: Cargando grafo optimizado persistido desde '{pkl_file}'...")
        G = ig.Graph.Read_Pickle(pkl_file)
        G = aplicar_regla_neutral_cortos(G)
        with open(topics_file, "r", encoding="utf-8") as f:
            topics = {int(k): v for k, v in json.load(f).items()}
        return G, topics
    else:
        print(f"[*] Construyendo grafo definitivo para r/{sub}...")
        sub_indices = np.flatnonzero(df_full["subreddit"] == sub)
        X_sub = np.array(emb_full[sub_indices], dtype=np.float32)
        X_sub -= X_sub.mean(axis=0)
        faiss.normalize_L2(X_sub)
        
        # Búsqueda FAISS
        dim = X_sub.shape[1]
        idx_flat = faiss.IndexFlatIP(dim)
        idx_flat.add(X_sub)
        D, I = idx_flat.search(X_sub, cfg["k"] + 1)
        
        from generar_grafos_finales import build_graph_from_neighbors
        G_base = build_graph_from_neighbors(I, D, len(sub_indices), cfg["k"], cfg["threshold"])
        
        # Leiden
        cl = G_base.community_leiden(objective_function="modularity", weights=G_base.es["weight"], resolution=cfg["resolution"])
        base_mem = np.array(cl.membership)
        sizes = cl.sizes()
        subst = [i for i, s in enumerate(sizes) if s >= 500]
        
        # Macro-clustering
        centroids = np.array([X_sub[base_mem == c].mean(axis=0) for c in subst])
        centroids /= (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-8)
        agg = AgglomerativeClustering(n_clusters=cfg["target_k"], metric="cosine", linkage="average")
        macro_labels = agg.fit_predict(centroids)
        c_to_macro = {subst[i]: macro_labels[i] for i in range(len(subst))}
        
        valid_nodes = np.flatnonzero([c_to_macro.get(m, -1) != -1 for m in base_mem])
        G = G_base.subgraph(valid_nodes)
        G.vs["community"] = [c_to_macro[base_mem[i]] for i in valid_nodes]
        G = aplicar_regla_neutral_cortos(G)
        G.write_pickle(pkl_file)
        return G, {}

grafos_cargados = {}
for sub, cfg in CONFIGURACIONES_GANADORAS.items():
    G, topics = cargar_o_construir_grafo(sub, cfg, df_full, emb_full)
    grafos_cargados[sub] = {"graph": G, "topics": topics}
    print(f"    r/{sub}: {G.vcount():,} nodos, {G.ecount():,} aristas, {len(set(G.vs['community']))} comunidades finales.")


[+] r/Anxiety: Cargando grafo optimizado persistido desde 'Grafo_Anxiety_best.pkl'...
    r/Anxiety: 274,293 nodos, 5,762,057 aristas, 10 comunidades finales.
[+] r/SuicideWatch: Cargando grafo optimizado persistido desde 'Grafo_SuicideWatch_best.pkl'...
    r/SuicideWatch: 464,961 nodos, 9,628,468 aristas, 10 comunidades finales.
[+] r/depression: Cargando grafo optimizado persistido desde 'Grafo_depression_best.pkl'...
    r/depression: 613,107 nodos, 12,760,422 aristas, 12 comunidades finales.


## Caracterización de Macrocomunidades
Para cada macrocomunidad final (entre 10 y 12 por subreddit), se extraen los descriptores temáticos mediante `c-TF-IDF` refinado, eliminando palabras espurias (slurs, typos, ruido de bots) y asegurando que las etiquetas identifiquen nítidamente la realidad psicológica de cada clúster.


In [21]:
for sub in SUBREDDITS_OBJETIVO:
    topics_file = f"topics_{sub}_best.json"
    if os.path.exists(topics_file):
        with open(topics_file, "r", encoding="utf-8") as f:
            topics = json.load(f)
        print(f"\nTópicos Semánticos r/{sub}:")
        for cid, words in sorted(topics.items(), key=lambda x: int(x[0])):
            print(f"  Macro C{cid}: {', '.join(words)}")



Tópicos Semánticos r/Anxiety:
  Macro C0: pfizer, vax, variant, moderna, workbook
  Macro C1: gpa, workload, unemployment, attendance, marketing
  Macro C2: ecg, gabapentin, mirtazapine, fluoxetine, cymbalta
  Macro C3: meow, leash, crate, purr, rehome
  Macro C4: turbulence, cockroach, pedestrian, flying, dmv
  Macro C5: talkspace, confidentiality, likey, medicaid, sandra
  Macro C6: vu, deja, schizo, hypnagogic, asteroid
  Macro C7: instrumental, lofi, indie, binaural, cello
  Macro C8: erection, bridesmaid, bride, propose, flirty
  Macro C9: insta, deactivate, minecraft, inferior, unworthy

Tópicos Semánticos r/SuicideWatch:
  Macro C0: scholarship, counsellor, cbt, supervisor, academic_pressure
  Macro C1: isolation, holidays, recruitment, hopeless, loneliness
  Macro C2: lethal_methods, gunshot, asphyxiation, hanging, overdose
  Macro C3: discord, cyberbullying, screenshot, follower, digital_isolation
  Macro C4: zoloft, prozac, wellbutrin, klonopin, acetaminophen
  Macro C5: cri

# Visualización

### Muestreo Estratificado con Alineación Temática al Tópico 
Para garantizar que los comentarios visibles en la vista micro ejemplifiquen con máxima claridad el tópico de cada comunidad:
1. **Alineación Temática (50%):** Se priorizan comentarios que reflejan directamente el núcleo semántico del clúster y contienen términos distintivos del tópico.
2. **Diversidad Afectiva Condicionada (50%):** Se seleccionan comentarios que expresan las emociones observadas en el grupo (`anger`, `fear`, `joy`, `sadness`, `surprise`, etc.) condicionados a una representatividad temática sólida.
3. **Preservación Íntegra:** Textos completos con formato legible, IDs originales y sin etiquetas patologizantes.


In [22]:
from visualizador_redes_sna import sample_representative_subgraph

# Demostración del muestreo en r/Anxiety
G_anx = grafos_cargados["Anxiety"]["graph"]
topics_anx = grafos_cargados["Anxiety"]["topics"]
G_sample_anx = sample_representative_subgraph(G_anx, topics_anx, max_nodes_total=500, random_seed=SEMILLA)

print(f"Muestra estratificada generada: {G_sample_anx.vcount()} nodos.")
print("Distribución de emociones en la muestra representativa:")
display(pd.Series(G_sample_anx.vs["factor"]).value_counts().to_frame("Comentarios"))


Muestra estratificada generada: 500 nodos.
Distribución de emociones en la muestra representativa:


,Comentarios
fear,154
surprise,77
sadness,70
joy,67
neutral,58
anger,42
disgust,32


## Generación del Dashboard Analítico Interactivo SNA 



In [23]:
from generar_dashboard_sna_html import generar_dashboard_completo

print("[*] Ensamblando Dashboard Analítico Interactivo V2.1...")
generar_dashboard_completo(output_html="dashboard_grafos_interactivos_sna.html")
print("[OK] Dashboard compilado exitosamente. Listo para visualización en navegador.")


[*] Ensamblando Dashboard Analítico Interactivo V2.1...

GENERADOR DEL DASHBOARD ANALÍTICO INTERACTIVO SNA V2.2 (5 PESTAÑAS)

[*] Procesando r/Anxiety para Dashboard SNA (5 pestañas optimizadas)...
    -> Generando Pestaña 1: Grafo Micro Orgánico (Filtro Trimestral y Totales)...
    -> Generando Pestaña 2: Grafo Macro Agregado Sin Filtrar (Todas las Comunidades Conectadas)...
    -> Generando Pestaña 3: Evolución Temporal del Grafo Macro Sin Filtrar (Total por Periodo y Controles)...
    -> Generando Pestaña 4: Matriz de Burbujas (Orden Cronológico Estricto)...
    -> Generando Pestaña 5: Volumen Longitudinal por Comunidad (Dual)...
    [OK] r/Anxiety procesado y guardado en caché en 141.9s.

[*] Procesando r/SuicideWatch para Dashboard SNA (5 pestañas optimizadas)...
    -> Generando Pestaña 1: Grafo Micro Orgánico (Filtro Trimestral y Totales)...
    -> Generando Pestaña 2: Grafo Macro Agregado Sin Filtrar (Todas las Comunidades Conectadas)...
    -> Generando Pestaña 3: Evolución Te